In [20]:
import pandas as pd
import numpy as np

In [21]:
df = pd.read_csv(r'C:\Users\mabin\Desktop\DataScienceClassNotes\Spam_Classifier_Project\cleaned_spam.csv')

In [22]:
df.head()

,target,text,num_characters,num_words,num_sentences
0,0,"Go until jurong point, crazy.. Available only ...",111,24,2
1,0,Ok lar... Joking wif u oni...,29,8,2
2,1,Free entry in 2 a wkly comp to win FA Cup fina...,155,37,2
3,0,U dun say so early hor... U c already then say...,49,13,1
4,0,"Nah I don't think he goes to usf, he lives aro...",61,15,1


## Data Preprocessing

In [23]:
import contractions
import re
import spacy
from bs4 import BeautifulSoup
# -- The Comprehensive Normalization Pipline --

nlp = spacy.load("en_core_web_sm")

def normilization_pipeline(text):
    """
    A more realistic pipeline to clean scraped data
    """
    
    # --- Step 1: Structural Cleaning ---
    # Remove HTML tags using BeautifulSoup
    text = BeautifulSoup(text, "html.parser").get_text()
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    
    # Remove email addresses
    text = re.sub(r'\S*@\S*\s?', '', text)
    
    # Removing special characters
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    
    
    
    # --- Step 2: Linguistic Normalization (Pre-tokenization) ---
    # Expand contractions
    text = contractions.fix(text)
    
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    
    
    
    # --- Step 3: Token-based Normalization using SpaCy ---
    doc = nlp(text)
    
    clean_tokens = [
        token.lemma_.lower() for token in doc
        if not token.is_stop and not token.is_punct and not token.is_space
    ]
    
    clean_tokens = " ".join(clean_tokens) # To convet list into str
    # print(text)
    return clean_tokens

In [24]:
df['transformed_text'] = df['text'].apply(normilization_pipeline)

In [25]:
df.head()

,target,text,num_characters,num_words,num_sentences,transformed_text
0,0,"Go until jurong point, crazy.. Available only ...",111,24,2,jurong point crazy available bugis n great wor...
1,0,Ok lar... Joking wif u oni...,29,8,2,ok lar joking wif oni
2,1,Free entry in 2 a wkly comp to win FA Cup fina...,155,37,2,free entry wkly comp win fa cup final tkts st ...
3,0,U dun say so early hor... U c already then say...,49,13,1,dun early hor c
4,0,"Nah I don't think he goes to usf, he lives aro...",61,15,1,nah think go usf live


## Model building

As we know the best model will be the Naive Bayes MultinominalNB()

In [26]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=3000)


In [27]:
X = tfidf.fit_transform(df['transformed_text']).toarray()

In [28]:
y = df['target'].values

In [29]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=2)

In [30]:
# importing model and metrics
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score,confusion_matrix,precision_score

In [31]:
mnb = MultinomialNB()

In [32]:
mnb.fit(X_train,y_train)
y_pred2 = mnb.predict(X_test)
print(accuracy_score(y_test,y_pred2))
print(confusion_matrix(y_test,y_pred2))
print(precision_score(y_test,y_pred2))

0.971953578336557
[[896   0]
 [ 29 109]]
1.0


## Exporting required materials for website building

In [33]:
import pickle

pickle.dump(tfidf,open('vectorizer.pkl','wb'))
pickle.dump(mnb,open('model.pkl','wb'))